In [1]:
from langchain_openrouter import ChatOpenRouter
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
from langchain.tools import tool

load_dotenv(dotenv_path="../.env")

/home/lamhung/code/external_knowledge_base/backend/.venv/lib/python3.14/site-packages/langchain_core/utils/pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
/home/lamhung/code/external_knowledge_base/backend/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
base_url = "https://openrouter.ai/api/v1"
from langchain.agents import create_agent

model = "openrouter:nvidia/nemotron-3.5-lightning:free"

In [3]:
from langchain.chat_models import init_chat_model

In [4]:
llm = init_chat_model(
    model=model, api_key=os.getenv("OPENROUTER_API_KEY"), base_url=base_url
)

In [5]:
@tool
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [6]:
agent = create_agent(
    model=llm,
    tools=[get_weather],
)

In [7]:
response = agent.astream(
    {"messages": [{"role": "user", "content": "What is weather in Ha Noi?"}]}
)

In [8]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is weather in Ha Noi?"}]},
    version="v3",
)
thingking, tool, answer = False, False, False
for message in stream.messages:
    for delta in message.reasoning:
        if not thingking:
            print("\nThinking: ", end="", flush=True)
            thingking = True
            tool = False
            answer = False
        print(f"{delta}", end="", flush=True)
    for delta in message.tool_calls:
        if not tool:
            print("\nUsing tool: ", end="", flush=True)
            tool = True
            thingking = False
            answer = False
        print(f"{delta}", end="", flush=True)

    for delta in message.text:
        if answer == False:
            print("\nAnswer: ", end="", flush=True)
            answer = True
            thingking = False
            tool = False
        print(delta, end="", flush=True)

/home/lamhung/code/external_knowledge_base/backend/.venv/lib/python3.14/site-packages/langgraph/pregel/main.py:3708: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return self._pregel_stream_v3(
/home/lamhung/code/external_knowledge_base/backend/.venv/lib/python3.14/site-packages/langgraph/pregel/main.py:3558: LangChainBetaWarning: The v3 streaming protocol on Pregel is experimental.
  return GraphRunStream(graph_iter, mux)


KeyboardInterrupt: 

In [ ]:
out = stream.output

In [ ]:
out

In [ ]:
out["messages"][-1].content[-2]

In [ ]:
out["messages"][-1].content[-1]

In [ ]:
type(out["messages"][-1].content[-1])

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(model=llm, tools=[get_weather], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": str(uuid7())}}
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    config=config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nTool result: {item.output}")

final_state = stream.output

In [ ]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"


agent = create_agent(
    model=llm,
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        print(f"node: {metadata['langgraph_node']}")
        print(f"content: {token.content_blocks}")
        print("\n")

In [ ]:
from langchain.agents import create_agent
from langchain_core.utils.uuid import uuid7
from langgraph.checkpoint.memory import InMemorySaver


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_agent(model=llm, tools=[get_weather], checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": str(uuid7())}}
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    config=config,
    version="v3",
)
for kind, item in stream.interleave("messages", "tool_calls"):
    if kind == "messages":
        for token in item.text:
            print(token, end="", flush=True)
    elif kind == "tool_calls":
        print(f"\nTool call: {item.tool_name}({item.input})")
        for delta in item.output_deltas:
            print(delta, end="", flush=True)
        print(f"\nTool result: {item.output}")

final_state = stream.output

In [ ]:
k = final_state["messages"][1]

In [ ]:
td = k.to_json()

In [ ]:
type(td)

In [ ]:
import json

In [ ]:
with open("output.json", "w") as f:
    f.write(json.dumps(td))

In [ ]:
config = {"configurable": {"thread_id": str(uuid7())}}

In [ ]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    version="v3",
    config=config,
)

for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call chunk: {chunk}")

    # finalized = message.tool_calls.get()
    # if finalized:
    #     print(f"finalized tool calls: {finalized}")

In [ ]:
for call in stream.tool_calls:
    print("hi")
    print(f"{call.tool_name}({call.input})")
    for delta in call.output_deltas:
        print(delta, end="", flush=True)
    print(call.output, call.error)

In [ ]:
from tracemalloc import stop

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


weather_agent = create_agent(
    model=llm,
    tools=[get_weather],
    name="weather_agent",
)


def call_weather(query: str) -> str:
    """Query the weather agent."""
    result = weather_agent.invoke({"messages": [{"role": "user", "content": query}]})
    return result["messages"][-1].text


supervisor = create_agent(
    model=llm,
    tools=[call_weather],
    name="supervisor",
)

stream = supervisor.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Boston?"}]},
    version="v3",
)
for message in stream.messages:
    for chunk in message.tool_calls:
        print(f"tool call: {chunk}")
    for delta in message.reasoning:
        print(f"[thinking] {delta}", end="", flush=True)
    for delta in message.text:
        print(f"[text] {delta}", end="", flush=True)
# for subagent in stream.subagents:
#     print(f"{subagent.name}: ", end="")
#     for message in subagent.messages:
#         for token in message.text:
#             print(token, end="", flush=True)
#     print()

In [ ]:
rep = stream.output

In [ ]:
rep["messages"]

In [ ]:
stream = agent.stream_events(
    {"messages": [{"role": "user", "content": "What's the weather in Boston?"}]},
    version="v3",
    config=config,
)

for name, item in stream.interleave("messages", "tool_calls", "values"):
    if name == "messages":
        print(item.text)
    elif name == "tool_calls":
        print(item.tool_name, item.input)
    elif name == "values":
        print(item)

In [ ]:
reasoning, tool_call, text = False, False, False

In [ ]:
run = agent.stream_events(
    {
        "messages": [
            {"role": "user", "content": "Tìm thời tiết Hà Nội và giải thích cho tôi"}
        ]
    },
    config=config,
    version="v3",
)

for message in run.messages:
    # Text thông thường
    if text == False:
        print("\n[TEXT]: ", end="", flush=True)
        text = True
        reasoning = False
        tool_call = False

    for delta in message.text:
        print(delta, end="", flush=True)

    # Reasoning
    if reasoning == False:
        print("\n[REASONING]: ", end="", flush=True)
        reasoning = True
        text = False
        tool_call = False
    for delta in message.reasoning:
        print(delta, end="", flush=True)

    # Tool call
    if tool_call == False:
        print("\n[TOOL CALL]: ", end="", flush=True)
        tool_call = True
        reasoning = False
        text = False
    for tool_call in message.tool_calls:
        print(tool_call, end="", flush=True)

    # Message hoàn chỉnh
    # output = message.output
    # print("\n[OUTPUT]", output)

In [ ]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"The weather in {city} is sunny, 30°C."


agent = create_agent(
    model=llm,
    tools=[get_weather],
)


run = agent.stream_events(
    {"messages": [{"role": "user", "content": "What is the weather in Hanoi?"}]},
    version="v3",
)

reasoning, tool_call, text = False, False, False
# =========================
# 1. LLM MESSAGE STREAM
# =========================

for message in run.messages:
    # -------------------------
    # Reasoning
    # -------------------------

    for delta in message.reasoning:
        if reasoning == False:
            print("\n[REASONING]: ", end="", flush=True)
            reasoning = True
            text = False
            tool_call = False
        print(delta, end="", flush=True)

    # -------------------------
    # Normal text
    # -------------------------
    for delta in message.text:
        if text == False:
            print("\n[TEXT]: ", end="", flush=True)
            text = True
            reasoning = False
            tool_call = False
        print(delta, end="", flush=True)

    # -------------------------
    # Tool call
    # -------------------------
    for tool_call in message.tool_calls:
        if tool_call == False:
            print("\n[TOOL CALL]: ", end="", flush=True)
            tool_call = True
            reasoning = False
            text = False
        print(tool_call, end="", flush=True)


# =========================
# 2. TOOL EXECUTION / RESULT
# =========================

for call in run.tool_calls:
    print("[TOOL]", call.tool_name)
    print("[INPUT]", call.input)

    # Tool output streaming
    for delta in call.output_deltas:
        print("[TOOL RESULT DELTA]", delta, flush=True)

    # Final tool result
    print("[TOOL RESULT]", call.output)

    # Error nếu tool fail
    if call.error:
        print("[TOOL ERROR]", call.error)


# =========================
# 3. FINAL OUTPUT
# =========================

print("[FINAL]", run.output)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import (
    AIMessage,
    AIMessageChunk,
    ToolMessage,
)


def get_weather(city: str) -> str:
    """Get the weather for a city."""
    return f"The weather in {city} is sunny, 30°C."


agent = create_agent(
    model=llm,
    tools=[get_weather],
)

reasoning, tool_call, text = False, False, False

for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if not isinstance(token, AIMessageChunk):
            continue
        # -------------------------------------------------
        # REASONING
        # -------------------------------------------------

        for block in token.content_blocks:
            if block["type"] == "reasoning":
                if reasoning == False:
                    print("\n[REASONING]: ", end="", flush=True)
                    reasoning = True
                    text = False
                    tool_call = False
                print(block["reasoning"], flush=True, end="")

        # -------------------------------------------------
        # NORMAL TEXT
        # -------------------------------------------------

        if token.text:
            if text == False:
                print("\n[TEXT]: ", end="", flush=True)
                text = True
                reasoning = False
                tool_call = False
            print(token.text, flush=True, end="")

        # -------------------------------------------------
        # TOOL CALL STREAM
        # -------------------------------------------------

        if token.tool_call_chunks:
            for tool_call in token.tool_call_chunks:
                if tool_call == False:
                    print("\n[TOOL CALL CHUNK]", end="", flush=True)
                    tool_call = True
                    reasoning = False
                    text = False
                print(tool_call, flush=True, end="")

    # =====================================================
    # 2. COMPLETED MESSAGE / TOOL RESULT
    # =====================================================

    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            message = update["messages"][-1]

            # -------------------------------------------------
            # COMPLETED AI MESSAGE
            # -------------------------------------------------

            if isinstance(message, AIMessage):
                # Completed tool call
                if message.tool_calls:
                    for tool_call in message.tool_calls:
                        print(
                            "[TOOL CALL]",
                            tool_call,
                            flush=True,
                        )

            # -------------------------------------------------
            # TOOL RESULT
            # -------------------------------------------------

            elif isinstance(message, ToolMessage):
                print(
                    "[TOOL RESULT]",
                    message.content,
                    flush=True,
                )

In [ ]:
from langchain.agents import create_agent
from langchain.messages import AIMessage, ToolMessage


def get_weather(city: str) -> str:
    """Get weather for a city."""
    return f"The weather in {city} is sunny, 30°C."


agent = create_agent(
    model=llm,
    tools=[get_weather],
)


# Buffer cho từng AI turn
reasoning_buffer = []
text_buffer = []
tool_call_buffer = {}


for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    # =====================================================
    # MESSAGES: realtime chunks
    # =====================================================

    if chunk["type"] == "messages":
        token, metadata = chunk["data"]

        # -----------------------------
        # Reasoning
        # -----------------------------

        for block in token.content_blocks:
            if block["type"] == "reasoning":
                reasoning = block.get("reasoning", "")

                if reasoning:
                    reasoning_buffer.append(reasoning)

        # -----------------------------
        # Normal text
        # -----------------------------

        if token.text:
            text_buffer.append(token.text)

        # -----------------------------
        # Tool call chunks
        # -----------------------------

        for tc in token.tool_call_chunks:
            index = tc.get("index", 0)

            if index not in tool_call_buffer:
                tool_call_buffer[index] = {
                    "name": "",
                    "args": "",
                    "id": None,
                }

            if tc.get("name"):
                tool_call_buffer[index]["name"] += tc["name"]

            if tc.get("args"):
                tool_call_buffer[index]["args"] += tc["args"]

            if tc.get("id"):
                tool_call_buffer[index]["id"] = tc["id"]

    # =====================================================
    # UPDATES: message completed
    # =====================================================

    elif chunk["type"] == "updates":
        for source, update in chunk["data"].items():
            message = update["messages"][-1]

            # =============================================
            # AI MESSAGE COMPLETED
            # =============================================

            if isinstance(message, AIMessage):
                # Reasoning - print ONCE
                if reasoning_buffer:
                    print("\n🧠 REASONING")
                    print("".join(reasoning_buffer))

                    reasoning_buffer.clear()

                # Tool call - print ONCE
                if message.tool_calls:
                    print("\n🔧 TOOL CALL")

                    for tool_call in message.tool_calls:
                        print(f"Tool: {tool_call['name']}")

                        print(f"Args: {tool_call['args']}")

                    tool_call_buffer.clear()

                # Normal text - print ONCE
                if text_buffer:
                    print("\n💬 TEXT")
                    print("".join(text_buffer))

                    text_buffer.clear()

            # =============================================
            # TOOL RESULT
            # =============================================

            elif isinstance(message, ToolMessage):
                print("\n📦 TOOL RESULT")
                print(message.content)

In [ ]:
from langchain.messages import AIMessage, AIMessageChunk, ToolMessage


reasoning_buffer = []
text_buffer = []
tool_call_buffer = {}

storage = []
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    # =====================================================
    # MESSAGES
    # =====================================================
    print(chunk)
    storage.append(chunk)

In [ ]:
storage[4]

In [ ]:
type(storage[4]["data"])

In [ ]:
len(storage[4]["data"])

In [ ]:
storage[4]["data"]

In [ ]:
type(storage[4]["data"][0])

In [ ]:
storage[4]["data"][0]

In [ ]:
from langchain.messages import AIMessage, AIMessageChunk, ToolMessage


reasoning_buffer = []
text_buffer = []
tool_call_buffer = {}
storage = []
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    storage.append(chunk)
    print(chunk)

In [ ]:
from langchain.messages import AIMessage, AIMessageChunk, ToolMessage

textb, reasoningb, toolcallb = False, False, False
reasoning_buffer = []
text_buffer = []
tool_call_buffer = {}
storage = []
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        if isinstance(token, AIMessageChunk):
            if token.additional_kwargs.get("reasoning_content"):
                if reasoningb == False:
                    print("\n[REASONING]: ", end="", flush=True)
                    reasoningb = True
                    textb = False
                    toolcallb = False
                print(
                    token.additional_kwargs.get("reasoning_content"), end="", flush=True
                )
            if token.tool_call_chunks:
                if toolcallb == False:
                    print("\n[TOOL CALL]: ", end="", flush=True)
                    toolcallb = True
                    reasoningb = False
                    textb = False
                print(token.tool_call_chunks, end="", flush=True)
            elif token.text:
                if textb == False:
                    print("\n[TEXT]: ", end="", flush=True)
                    textb = True
                    reasoningb = False
                    toolcallb = False
                print(token.text, end="", flush=True)
        if isinstance(token, ToolMessage):
            print("\n[TOOL CALL RESULT]: ", end="", flush=True)
            print(token.content, end="", flush=True)

In [ ]:
from modules.llm.ag_ui_converter import LangChainAGUIAdapter

In [ ]:
adapter = LangChainAGUIAdapter()

In [ ]:
for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What is the weather in Hanoi?",
            }
        ]
    },
    stream_mode=["messages", "updates"],
    version="v2",
):
    adapter.transform(chunk=chunk)

In [ ]:
storage[8]["data"][0]

In [ ]:
type(storage[8]["data"][0])

In [ ]:
storage[-1]

In [ ]:
storage[-1]["data"]

In [ ]:
type(storage[-1]["data"])

In [ ]:
storage[-1]["data"].keys()

In [ ]:
storage[-1]["data"]["model"]

In [ ]:
storage[-1]["data"]["model"]["messages"]

In [ ]:
storage[-1]["data"]["model"]["messages"][0]